## Analyze results low-quality mapping

In [ ]:
import scanpy as sc
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../../../")
sys.path.append("../../utils/")
import mapqc
import data_cleanup
import analysis

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

params

In [ ]:
study = "Sheppard_2020"
batch_variable = "dataset"
testrun = False
use_old_counts = False
kl = 1.0
incl_SCD = False

paths:

In [ ]:
if testrun:
    suffix = "_test"
    suffix_2 = ""
elif incl_SCD:
    suffix = "_incl_SCD"
else:
    suffix = ""
if use_old_counts == True:
    suffix_2 = "original_counts_"
else:
    suffix_2 = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_{suffix_2}batch_{batch_variable}_kl{kl}_latent{suffix}.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_counts{suffix}.h5ad"
path_mapqc_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix_2}batch_{batch_variable}_kl{kl}{suffix}/mapqc"

load relevant files:

In [ ]:
path_counts

In [ ]:
path_emb

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

set gene symbols as adata.var row names (instead of IDs):

In [ ]:
adata.var["ensembl_id"] = adata.var.index
adata.var.index = [
    fn if not pd.isnull(fn) else gid
    for fn, gid in zip(adata.var.feature_name, adata.var.ensembl_id)
]

In [ ]:
adata.X[-6:, -6:].toarray()

In [ ]:
results_healthy = mapqc.load(os.path.join(path_mapqc_results_dir, "control.pkl"))
results_disease = mapqc.load(os.path.join(path_mapqc_results_dir, "case.pkl"))

correct annotation hierarchy mistake in HLCA: pericytes are said to be part of the fibroblast supergroup, when they are actually a separate cell type

In [ ]:
for lev in [2, 3]:
    for ann_colname in [f"ann_level_{lev}", f"ann_level_{lev}_pred"]:
        adata.obs[ann_colname] = adata.obs[ann_colname].tolist()
        results_healthy.adata.obs[ann_colname] = results_healthy.adata.obs[
            ann_colname
        ].tolist()
        results_disease.adata.obs[ann_colname] = results_disease.adata.obs[
            ann_colname
        ].tolist()
        if "_pred" in ann_colname:
            ref_col = "ann_level_5_pred"
        else:
            ref_col = "ann_level_5"
        adata.obs.loc[
            adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"
        results_healthy.adata.obs.loc[
            results_healthy.adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"
        results_disease.adata.obs.loc[
            results_disease.adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"

copy results to main adata:

In [ ]:
for stat in [
    "mapqc_score",
    "mapqc_score>2",
]:
    adata.obs[stat] = np.nan
    healthy_idc = [
        idx for idx in adata.obs.index if idx in results_healthy.adata.obs.index
    ]
    adata.obs.loc[healthy_idc, stat] = results_healthy.adata.obs.loc[healthy_idc, stat]
    disease_idc = [
        idx for idx in adata.obs.index if idx in results_disease.adata.obs.index
    ]
    adata.obs.loc[disease_idc, stat] = results_disease.adata.obs.loc[disease_idc, stat]

## Plots:

In [ ]:
FIGURES = dict()

UMAP colored by reference, query control, query case

In [ ]:
# create variable splitting data in ref control, query control
adata.obs["r_or_q_ctr_or_case"] = [
    f"{r_q} {stat}" for r_q, stat in zip(adata.obs.r_or_q, adata.obs.status)
]
# clean up:
cleaning_dict = {
    "r control": "Reference (controls)",
    "q control": "Query controls",
    "q IPF": "Query IPF",
}
adata.obs.r_or_q_ctr_or_case = adata.obs.r_or_q_ctr_or_case.map(cleaning_dict)
results_healthy.adata.obs["r_or_q_ctr_or_case"] = adata.obs.loc[
    results_healthy.adata.obs.index, "r_or_q_ctr_or_case"
].copy()
results_disease.adata.obs["r_or_q_ctr_or_case"] = adata.obs.loc[
    results_disease.adata.obs.index, "r_or_q_ctr_or_case"
].copy()
palette_1 = {
    "Reference (controls)": "grey",
    "Query controls": "dodgerblue",
    "Query IPF": "midnightblue",
}

In [ ]:
FIGURES["umap_r_or_q_ctr_or_case"] = sc.pl.umap(
    adata,
    color="r_or_q_ctr_or_case",
    frameon=False,
    ncols=1,
    palette=palette_1,
    show=False,
    s=0.7,
)

UMAP colored by cell type annotation:  

In [ ]:
for lev in range(1, 6):
    adata.obs[f"ann_level_{lev}_merged"] = [
        ann if not pd.isnull(ann) else ann_pred
        for ann, ann_pred in zip(
            adata.obs[f"ann_level_{lev}"], adata.obs[f"ann_level_{lev}_pred"]
        )
    ]

In [ ]:
FIGURES["umap_anns"], ax = plt.subplots(figsize=(8, 8))
sc.pl.umap(adata, color="ann_level_3_merged", frameon=False, ax=ax, s=0.7)

UMAP colored by mapqc score controls:

In [ ]:
adata.n_obs

In [ ]:
FIGURES["umap_mapqc_score"] = mapqc.pl.umap_mapqc_scores(
    results_healthy, return_fig=True
)

In [ ]:
FIGURES["umap_ctr_and_case_mapqc_score_binarized"], ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(
    adata,
    color="mapqc_score>2",
    # return_fig=True,
    size=0.5,
    palette={
        ">2": "firebrick",
        "<=2": "navajowhite",
        "filtered out": "black",
        "not sampled": "cornflowerblue",
    },
    frameon=False,
    na_color="gray",
    ax=ax,
)

In [ ]:
FIGURES["umap_ctr_mapqc_score_binarized"] = mapqc.pl.umap_mapqc_scores(
    results_healthy, color_by="mapqc_score>2", return_fig=True, size=0.5
)

In [ ]:
FIGURES["umap_case_mapqc_score_binarized"] = mapqc.pl.umap_mapqc_scores(
    results_disease, color_by="mapqc_score>2", return_fig=True
)

Boxplot showing mapqc scores in controls per cell type:

In [ ]:
FIGURES["mapqc_per_ct_ctr_only"] = mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy,
    box_category="ann_level_2_pred",
    figsize=(5, 4.5),
    point_size=0.3,
    color_category="donor_id",
    return_fig=True,
    min_cells_per_cat=150,
)

In [ ]:
FIGURES[
    "grouped_boxplot_lev2_ann_mapqc_score"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_2_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(5, 4),
    vmax=15,
    min_cells_per_cat=100,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_3_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(12, 5),
    vmax=15,
    min_cells_per_cat=50,
)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_4_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(12, 5),
    vmax=15,
    min_cells_per_cat=50,
)

### Analysis endothelial cells:

In [ ]:
FIGURES[
    "grouped_barplot_end_subtype_mapqc_scores"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_5_pred",
    group_category="r_or_q_ctr_or_case",
    box_groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(5, 5),
    palette=palette_1,
    min_cells_per_cat=50,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

subset to endothelial cells:

In [ ]:
adata_end = adata[adata.obs.ann_level_2_merged == "Blood vessels", :].copy()

re-run knn graph and umap

In [ ]:
sc.pp.neighbors(adata_end, n_neighbors=30, use_rep="X_scanvi_emb")
sc.tl.umap(adata_end)

In [ ]:
sc.tl.leiden(adata_end, key_added="leiden_end", resolution=0.3)

clean up EC annotations:

In [ ]:
adata_end.obs["ann_level_5_cleaned"] = adata_end.obs.ann_level_5_merged.copy()
end_ct_count = adata_end.obs.ann_level_5_cleaned.value_counts()
min_n_cells = 50
low_n_anns = end_ct_count.index[end_ct_count < min_n_cells]
renamer = {
    (ct): ("Other" if ct in low_n_anns else ct)
    for ct in adata_end.obs.ann_level_5_cleaned.unique()
}
adata_end.obs.ann_level_5_cleaned = adata_end.obs.ann_level_5_cleaned.map(renamer)
# palette_2 = {}
# cmap = plt.get_cmap("tab10").colors
# num_colors = len(cmap)

# for index, category in enumerate(adata_end.obs.ann_level_5_cleaned.unique()):
#     color = cmap[index % num_colors]
#     palette_2[category] = color
# palette_2["Other"] = "tab:grey"
palette_2 = {
    "EC aerocyte capillary": "#377eb8",  # "tab:blue",
    "EC arterial": "#ff7f00",  # "tab:orange",
    "EC general capillary": "#4daf4a",
    "EC venous pulmonary": "#f781bf",
    "EC venous systemic": "#a65628",
    "Other": "#a9a9a9",
}  # '#984ea3','#999999', '#e41a1c', '#dede00']

In [ ]:
with plt.rc_context({"figure.figsize": (2, 2)}):
    sc.pl.umap(
        adata_end, color="ann_level_5_cleaned", palette=palette_2, frameon=False, s=0.5
    )

In [ ]:
sc.pl.umap(
    adata_end,
    color=[
        "r_or_q",
        "mapqc_score>2",
        "lung_condition",
        "leiden_end",
    ],
    wspace=0.3,
    frameon=False,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    mapqc_results=results_healthy,
    box_category="ann_level_5_pred",
    groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(2, 4),
    color_high_median_boxes_red=True,
    min_cells_per_cat=50,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    mapqc_results=results_disease,
    box_category="ann_level_5_pred",
    color_category="sample",
    groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(4, 4),
    min_cells_per_cat=50,
    color_high_median_boxes_red=False,
)

show cluster make-up:

In [ ]:
FIGURES["EC_cluster_barplot"], ax = plt.subplots(figsize=(4, 2))
pd.crosstab(adata_end.obs.leiden_end, adata_end.obs.r_or_q_ctr_or_case).plot(
    kind="bar", stacked=True, color=palette_1, ax=ax
)
leg = plt.legend(loc=(1.01, 0.4), frameon=False, title="Cell source")
leg._legend_box.align = "left"
ax.spines[["right", "top"]].set_visible(False)
ax.set_ylabel("N cells")
ax.set_xlabel("Blood vessel cell cluster")
plt.show()

In [ ]:
FIGURES["EC_subtype_barplot"], ax = plt.subplots(figsize=(4, 2))
pd.crosstab(adata_end.obs.leiden_end, adata_end.obs.ann_level_5_cleaned).plot(
    kind="bar", stacked=True, ax=ax, color=palette_2
)
leg = plt.legend(loc=(1.01, 0), frameon=False, title="EC subtype")
leg._legend_box.align = "left"
ax.spines[["right", "top"]].set_visible(False)
ax.set_ylabel("N cells")
ax.set_xlabel("Endothelial cell cluster")
plt.show()

marker plot:

In [ ]:
EC_markers = {
    "EC": ["SPARCL1", "AQP1", "RNASE1"],
    "EC arterial": ["DKK2", "IGFBP3"],
    "EC aerocyte capillary": ["SOSTDC1"],
    "EC general capillary": ["IL7R", "FCN3", "MT1M"],
    "EC venous systemic": ["TPD52L1", "ZNF385D", "OLFM1"],
    "EC venous pulmonary": [
        "CPE",
        "C7_ENSG00000112936",
        "MMRN1",
    ],  # gene with long name is actually C7 but somehow it is named with ensembl id here...
}

In [ ]:
adata_end.obs["ct_r_or_q"] = [
    f"{ct} ({r_or_q})"
    for ct, r_or_q in zip(adata_end.obs.ann_level_5_cleaned, adata_end.obs.r_or_q)
]

In [ ]:
sc.pl.dotplot(
    adata_end, var_names=EC_markers, groupby="ct_r_or_q", standard_scale="var"
)

##### Identify genes associated with low mixing:

In [ ]:
sc.tl.rank_genes_groups(adata_end, groupby="leiden_end", reference="rest", groups=["6"])

In [ ]:
sc.tl.filter_rank_genes_groups(adata_end)

In [ ]:
genes_used_for_mapping = pd.read_csv(
    "../HLCA_reference_model/var_names.csv", index_col=0, header=None
)
id_to_name_mapper = {
    gid: gn for gid, gn in zip(adata.var.ensembl_id, adata.var.feature_name)
}
genes_used_for_mapping["gene_id"] = genes_used_for_mapping.index
genes_used_for_mapping.index = genes_used_for_mapping.gene_id.map(id_to_name_mapper)

In [ ]:
end_degs_filt = sc.get.rank_genes_groups_df(
    adata_end, key="rank_genes_groups_filtered", group="6"
).dropna(how="any")
end_degs_filt["used_during_mapping"] = end_degs_filt.names.isin(
    genes_used_for_mapping.index
)

Plot top 10 DEGs that pass standard filtering, and showing only those that were used during mapping. These include, among others, B cell genes ("IGKC","IGHA1","IGLC2"), smooth muscle gene ("MYL9"), stress gene ("FOSB"), EC subtype-specific genes ("NNMT", "ACKR1")

In [ ]:
adata_end.obs["leiden_end_r_or_q_ctr_or_case"] = [
    f"{cl} ({status})"
    for cl, status in zip(adata_end.obs.leiden_end, adata_end.obs.r_or_q_ctr_or_case)
]
# set groups with fewer than min_n_cells_per_group to Other:
group_counts = adata_end.obs.leiden_end_r_or_q_ctr_or_case.value_counts()
min_n_cells = 50
other_groups = group_counts[group_counts < 50].index.tolist()
mapping = {
    gr: (gr if gr not in other_groups else "Other")
    for gr in adata_end.obs.leiden_end_r_or_q_ctr_or_case.unique()
}
adata_end.obs.leiden_end_r_or_q_ctr_or_case = (
    adata_end.obs.leiden_end_r_or_q_ctr_or_case.map(mapping)
)

In [ ]:
top_filtered_degs = end_degs_filt.names.iloc[:30][
    end_degs_filt.used_during_mapping.iloc[:30]
]
print(f"{len(top_filtered_degs)} genes:")
print(top_filtered_degs)

In [ ]:
gene_dict = {
    "Ambient B cell": ["IGKC", "IGHA1", "IGLC2"],
    "Amb. smooth muscle": ["MYL9"],
    "Dissociation stress": ["FOSB"],
    "EC subtype specific": ["NNMT", "ACKR1", "SPRY1"],
    "Ribosomal": ["RPS4Y1"],
    "Other": ["PCAT19"],
}

In [ ]:
FIGURES["dotplot_end_IPF_cluster_degs"] = sc.pl.dotplot(
    adata_end,
    var_names=gene_dict,
    groupby="leiden_end",
    standard_scale="var",
    return_fig=True,
)
FIGURES["dotplot_end_IPF_cluster_degs"].show()

In [ ]:
FIGURES["dotplot_end_IPF_cluster_degs_split_by_status"] = sc.pl.dotplot(
    adata_end,
    var_names=gene_dict,
    groupby="leiden_end_r_or_q_ctr_or_case",
    standard_scale="var",
    return_fig=True,
)
FIGURES["dotplot_end_IPF_cluster_degs_split_by_status"].show()

check ambience of gene (note that this data consist of lineage-specific samples (i.e. stroma) and samples with all lineages, and that ambient RNA likely occurred in the "all samples")

In [ ]:
adata.obs["ann_level_2_r_or_q_ctr_or_case"] = [
    f"{ann} ({status})"
    for ann, status in zip(adata.obs.ann_level_2_merged, adata.obs.r_or_q_ctr_or_case)
]
adata.obs["ann_level_2_r_or_q_ctr_or_case"] = analysis.clean_up_obs_column(
    adata, "ann_level_2_r_or_q_ctr_or_case", min_n_cells_per_cat=200
)

In [ ]:
FIGURES["dotplot_all_cells_IPF_cluster_degs_split_by_status"] = sc.pl.dotplot(
    adata,
    var_names=gene_dict,
    groupby="ann_level_2_r_or_q_ctr_or_case",
    standard_scale="var",
    return_fig=True,
)
FIGURES["dotplot_all_cells_IPF_cluster_degs_split_by_status"].show()